In [1]:
# ─────────────────────────────────────────────
# Notebook 03 - Feature Engineering
# Goal: Transform and create new features 
# to improve ML model performance
# Steps:
# 1. Load cleaned data
# 2. Clip negative target values to 0
# 3. Create per-bed ratio feature
# 4. Encode categorical columns
# 5. Apply log transform to target
# 6. Save modeling-ready dataset
# ─────────────────────────────────────────────

import pandas as pd
import numpy as np
import os

df = pd.read_csv('../data/cleaned_hospital_data.csv')
print(df.shape)
print(df['year'].value_counts().sort_index())

(22745, 27)
year
2019    4638
2020    4529
2021    4557
2022    4515
2023    4506
Name: count, dtype: int64


In [2]:
# ─────────────────────────────────────────────
# Step 1 - Clip Negative Target Values
# A negative uncompensated care cost means the 
# hospital received more than expected — rare 
# but exists in the data (we saw min = -$887K)
# log1p breaks on negative numbers so we clip 
# them to 0 before transforming
# ─────────────────────────────────────────────

# Check how many negative values exist
negative_count = (df['Cost of Uncompensated Care'] < 0).sum()
print(f"Negative values in target: {negative_count}")

# Clip anything below 0 to 0
df['Cost of Uncompensated Care'] = df['Cost of Uncompensated Care'].clip(lower=0)

# Confirm no negatives remain
print(f"Negative values after clipping: {(df['Cost of Uncompensated Care'] < 0).sum()}")
print(f"New min value: ${df['Cost of Uncompensated Care'].min():,.0f}")

Negative values in target: 60
Negative values after clipping: 0
New min value: $0


In [3]:
# ─────────────────────────────────────────────
# Step 2 - Create Per-Bed Ratio Feature
# Raw uncompensated care dollars favor urban 
# hospitals simply because they're bigger
# Dividing by number of beds normalizes for 
# hospital size — this surfaces how much burden
# each hospital carries RELATIVE to its capacity
# This is where the rural story gets interesting
# ─────────────────────────────────────────────

# Avoid division by zero — some hospitals may 
# have 0 beds reported, so we use clip(lower=1)
df['Uncompensated_Care_Per_Bed'] = (
    df['Cost of Uncompensated Care'] / 
    df['Number of Beds'].clip(lower=1)
)

# Compare rural vs urban on the new ratio
per_bed_comparison = df.groupby('Rural Versus Urban')['Uncompensated_Care_Per_Bed'].median()
print("Median Uncompensated Care Per Bed:")
print(per_bed_comparison)

# Also check raw dollar comparison for reference
raw_comparison = df.groupby('Rural Versus Urban')['Cost of Uncompensated Care'].median()
print("\nMedian Raw Uncompensated Care (for comparison):")
print(raw_comparison)

Median Uncompensated Care Per Bed:
Rural Versus Urban
R    43010.734694
U    40146.973032
Name: Uncompensated_Care_Per_Bed, dtype: float64

Median Raw Uncompensated Care (for comparison):
Rural Versus Urban
R    1694988.0
U    5382958.5
Name: Cost of Uncompensated Care, dtype: float64


In [4]:
# ─────────────────────────────────────────────
# Step 3 - Encode Categorical Columns
# ML models can't work with text categories
# We need to convert them to numbers
# Rural Versus Urban: R=0, U=1
# Type of Control and Provider Type are already 
# numbers (codes) so we leave them as is
# ─────────────────────────────────────────────

# Map Rural/Urban text to binary 0/1
# R (Rural) = 0, U (Urban) = 1
df['Is_Urban'] = df['Rural Versus Urban'].map({'R': 0, 'U': 1})

# Confirm mapping worked
print(df['Is_Urban'].value_counts())
print(f"\nNull values in Is_Urban: {df['Is_Urban'].isnull().sum()}")

Is_Urban
0    12143
1    10602
Name: count, dtype: int64

Null values in Is_Urban: 0


In [5]:
# ─────────────────────────────────────────────
# Step 4 - Apply Log Transform to Target
# As discussed in EDA, the target is heavily 
# right-skewed. log1p compresses the scale so
# the model treats all hospitals fairly
# We save the original column too so we can 
# reverse predictions back to real dollars
# ─────────────────────────────────────────────

# Apply log1p to the clipped target
# log1p(x) = log(x + 1), safe for zeros
df['log_uncompensated_care'] = np.log1p(df['Cost of Uncompensated Care'])

print("Original target stats:")
print(f"  Mean:   ${df['Cost of Uncompensated Care'].mean():,.0f}")
print(f"  Median: ${df['Cost of Uncompensated Care'].median():,.0f}")

print("\nLog transformed target stats:")
print(f"  Mean:   {df['log_uncompensated_care'].mean():.4f}")
print(f"  Median: {df['log_uncompensated_care'].median():.4f}")

# ─────────────────────────────────────────────
# Step 5 - Save Modeling-Ready Dataset
# This is what notebook 04 will load
# It has all original columns plus our new ones:
# - Uncompensated_Care_Per_Bed
# - Is_Urban  
# - log_uncompensated_care (the target for modeling)
# ─────────────────────────────────────────────

df.to_csv('../data/modeling_data.csv', index=False)
print(f"\nModeling dataset saved!")
print(f"Shape: {df.shape}")
print(f"\nNew columns added:")
print([c for c in df.columns if c not in ['Provider CCN', 'Hospital Name', 'City', 'State Code']][-3:])

Original target stats:
  Mean:   $8,912,746
  Median: $2,894,337

Log transformed target stats:
  Mean:   14.7835
  Median: 14.8783

Modeling dataset saved!
Shape: (22745, 30)

New columns added:
['Uncompensated_Care_Per_Bed', 'Is_Urban', 'log_uncompensated_care']
